# 04 · PS2 — Chu trình chuyển tiền có thứ tự thời gian (HI-Small)

**Câu hỏi (`problem_statements.html`).** Khi tiền rời một tài khoản rồi quay lại chính nó qua một vòng các tài khoản khác, **theo đúng thứ tự thời gian**, đó có phải rửa tiền không? Typology: CYCLE.

**Phương pháp theo file:** SCC → liệt kê chu trình dài 3–6 (Johnson) → chỉ giữ chu trình tôn trọng thời gian (mỗi bước xảy ra sau bước trước). **Tín hiệu sơ bộ cần kiểm tra lại:** giao dịch trên chu trình thời gian ACH có 22% là rửa tiền, recall ≈ 5%.

**Chạy:** chạy 01 → 02 bản 15/11 trước, rồi *Run All*. Không phải chỉnh gì.

| Bước | Việc làm | [PS] / [bổ sung] |
|---|---|---|
| 1 | Trần recall từ `Patterns.txt`: tỷ lệ CYCLE dài 3–6, tăng nghiêm ngặt theo thời gian, toàn ACH. Chọn giới hạn thời lượng vòng **δ = p95 thời lượng CYCLE trên dev** | bổ sung (Kumar & Calders 2018; Blanuša et al.) |
| 2 | SCC trên đồ thị cặp của từng đoạn, giữ SCC ≥ 2 tài khoản; view **ACH** (chính) và **mọi phương thức** (mở rộng) | PS |
| 3 | Liệt kê chu trình đơn tĩnh (mỗi chu trình một lần, gốc tại đỉnh nhỏ nhất, như Johnson 1975) dài 2–12 trên ACH, 2–6 trên mọi phương thức; có giới hạn số chu trình và thời gian | PS + bổ sung |
| 4 | Với mỗi chu trình và **mọi cách xoay vòng**: có tồn tại dãy giao dịch thật với timestamp **tăng nghiêm ngặt** và thời lượng ≤ δ? Gắn cờ mọi giao dịch nằm trên ít nhất một dãy hợp lệ | PS |
| 5 | Đánh giá cấp giao dịch (mẫu số = file gốc của đoạn) so với Random, ACH-only, ACH ngẫu nhiên cùng số cảnh báo, chu trình tĩnh, "hai đầu cùng SCC"; cấp pattern (tìm lại trọn attempt CYCLE) | PS + bổ sung |
| 6 | Kết luận: **Đào sâu** nếu cận dưới Wilson > ACH-only và precision > chu trình tĩnh và > SCC trên test, ổn định trên dev, ≥ 20 giao dịch dương; **Điều chỉnh** nếu chỉ thắng ACH-only; **Dừng** nếu không | R13 |

Chu trình phải nằm gọn trong một đoạn (dev / test / tail). Rule chính: **ACH, dài 3–6, tăng nghiêm ngặt, ≤ δ**. Các biến thể "không giảm (≤)", "không giới hạn δ", dài 2, dài 7–12 và view mọi phương thức chỉ để báo cáo độ nhạy.

In [1]:
import hashlib, json, math, os, re, time
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

print("04_ps2 · BẢN 15/11/2026 · kiểm tra file đầu vào...", flush=True)
ROOT = Path(os.environ["HI_SMALL_ROOT"]) if os.environ.get("HI_SMALL_ROOT") else Path(r"D:\KLTN\IBM_HI_Small_EDA")
PATTERNS_FILE = Path(os.environ.get("HI_SMALL_PATTERNS", r"D:\KLTN\HI-Small_Patterns.txt"))
EDGES_FILE = ROOT / "data" / "processed" / "graph" / "graph_edges.parquet"
TXN_FILE = ROOT / "data" / "processed" / "hi_small_graph_transactions.parquet"
REMOVED_FILE = ROOT / "data" / "processed" / "hi_small_removed_transactions.parquet"
PERIOD_FILE = ROOT / "reports" / "tables" / "analysis_period.json"
OUT = ROOT / "reports" / "ps2"
OUT.mkdir(parents=True, exist_ok=True)

NB01, NB02 = r"D:\KLTN\01_eda_hi_small.ipynb", r"D:\KLTN\02_build_graph_hi_small_duckdb.ipynb"
need = {TXN_FILE: NB01, REMOVED_FILE: NB01, PERIOD_FILE: NB01, EDGES_FILE: NB02, PATTERNS_FILE: "đặt HI-Small_Patterns.txt vào D:\\KLTN"}
missing = [f"  - {f}  → {nb}" for f, nb in need.items() if not f.is_file()]
if not missing:
    cols = {r[0] for r in duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{EDGES_FILE.as_posix()}')").fetchall()}
    if not {"from_bank_id", "usd_paid", "usd_recv"} <= cols:
        missing.append(f"  - {EDGES_FILE.name} là bản cũ → chạy lại {NB02}")
if missing:
    raise FileNotFoundError("Chưa đủ đầu vào cho PS2:\n" + "\n".join(missing))
print("Đủ file đầu vào.", flush=True)

CFG = {
    "main_view": "ACH",
    "views": {"ACH": {"l_max": 12, "time_limit_s": 900}, "ALL": {"l_max": 6, "time_limit_s": 300}},
    "main_lengths": [3, 6],
    "delta_quantile": 0.95,
    "max_cycles": 2_000_000,        # per view and segment
    "per_root_cap": 2_000,          # cycles reported from one root account (hub guard)
    "min_positives": 20,
    "seed": 42,
}
period = json.loads(PERIOD_FILE.read_text(encoding="utf-8"))
SEGMENTS = ["dev", "test", "tail"]
seg_bounds = np.array([int(pd.Timestamp(period[k]).timestamp()) for k in ("main_start", "dev_end", "main_end", "data_end")])
display(pd.DataFrame({"segment": SEGMENTS, "start": [period["main_start"], period["dev_end"], period["main_end"]],
                      "end (exclusive)": [period["dev_end"], period["main_end"], period["data_end"]]}))
db = duckdb.connect()
_ = db.execute("SET threads = 4")

04_ps2 · BẢN 15/11/2026 · kiểm tra file đầu vào...
Đủ file đầu vào.


,segment,start,end (exclusive)
0,dev,2022-09-01 00:00:00,2022-09-06 13:00:00
1,test,2022-09-06 13:00:00,2022-09-11 00:00:00
2,tail,2022-09-11 00:00:00,2022-09-18 16:19:00


## 1. Trần recall từ `Patterns.txt` và chọn δ (chỉ dùng dev)
Ghép từng dòng của `HI-Small_Patterns.txt` với `transaction_id` theo 10 cột gốc (chuỗi nguyên bản). Với mỗi attempt CYCLE: độ dài (số giao dịch), có **tăng nghiêm ngặt** theo thời gian dọc vòng không (xếp theo thời gian thì nối liền thành vòng, không có hai bước cùng phút), có toàn ACH không, thời lượng (giao dịch cuối − đầu), đoạn chứa giao dịch đóng vòng.

**δ** = phân vị 95% thời lượng của các attempt CYCLE tôn trọng thời gian **đóng vòng trong dev**, làm tròn lên theo giờ. `Patterns.txt` là nhãn, nên không dùng test hay tail để chọn δ.

In [2]:
BEGIN = re.compile(r"^BEGIN LAUNDERING ATTEMPT - (?P<typ>[A-Z-]+)")
END = re.compile(r"^END LAUNDERING ATTEMPT")
rows, attempt, typ, seq = [], -1, None, 0
with open(PATTERNS_FILE, encoding="utf-8-sig") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        m = BEGIN.match(line)
        if m:
            attempt += 1; typ = m.group("typ"); seq = 0; continue
        if END.match(line):
            typ = None; continue
        f = line.split(",")
        rows.append([attempt, typ, seq] + f[:10]); seq += 1
pat = pd.DataFrame(rows, columns=["attempt_id", "typology", "seq", "ts_raw", "from_bank", "from_acct", "to_bank",
                                  "to_acct", "amt_received_raw", "recv_ccy", "amt_paid_raw", "pay_ccy", "payment_format"])
pat["timestamp"] = pd.to_datetime(pat.ts_raw, format="%Y/%m/%d %H:%M")
db.register("pat", pat.drop(columns="ts_raw"))
pmap = db.execute("""
    SELECT p.attempt_id, p.typology, p.seq, min(t.transaction_id) AS transaction_id
    FROM pat p JOIN read_parquet(?) t
      ON t.is_laundering = 1 AND t.timestamp = p.timestamp
     AND t.from_bank = p.from_bank AND t.from_acct = p.from_acct
     AND t.to_bank = p.to_bank AND t.to_acct = p.to_acct
     AND t.amt_paid_raw = p.amt_paid_raw AND t.pay_ccy = p.pay_ccy
     AND t.amt_received_raw = p.amt_received_raw AND t.recv_ccy = p.recv_ccy
     AND t.payment_format = p.payment_format
    GROUP BY 1, 2, 3
""", [str(TXN_FILE)]).df()
print(f"Pattern rows: {len(pat):,}; matched to graph transactions: {len(pmap):,} "
      f"(the rest are self-loops/duplicates outside the graph)")
pmap = pmap.merge(pat[["attempt_id", "seq", "timestamp", "from_bank", "from_acct", "to_bank", "to_acct", "payment_format"]],
                  on=["attempt_id", "seq"])
pmap.to_csv(OUT / "ps2_pattern_transaction_map.csv", index=False)

def seg_of(ts_seconds):
    return np.searchsorted(seg_bounds, ts_seconds, side="right") - 1

def cycle_profile(g):
    g = g.sort_values(["timestamp", "seq"])
    src = (g.from_bank.astype(int).astype(str) + "_" + g.from_acct).tolist()
    dst = (g.to_bank.astype(int).astype(str) + "_" + g.to_acct).tolist()
    t = g.timestamp.tolist()
    chained = all(dst[i] == src[i + 1] for i in range(len(g) - 1)) and dst[-1] == src[0]
    strict = chained and all(t[i] < t[i + 1] for i in range(len(g) - 1))
    secs = g.timestamp.astype("datetime64[s]").astype(np.int64).to_numpy()
    segs = seg_of(secs)
    return pd.Series({"length": len(g), "time_respecting_strict": strict,
                      "time_respecting_nonstrict": chained, "all_ach": bool((g.payment_format == "ACH").all()),
                      "duration_h": (secs.max() - secs.min()) / 3600, "segment": int(segs.max()),
                      "single_segment": bool(segs.min() == segs.max())})

_cyc_rows = pmap[pmap.typology == "CYCLE"]
if len(_cyc_rows):
    cyc_pat = _cyc_rows.groupby("attempt_id")[_cyc_rows.columns.drop("attempt_id")].apply(cycle_profile)
else:
    cyc_pat = pd.DataFrame(columns=["length", "time_respecting_strict", "time_respecting_nonstrict", "all_ach",
                                    "duration_h", "segment", "single_segment"])
    print("⚠ Không ghép được attempt CYCLE nào từ Patterns.txt.")
cyc_pat["segment_name"] = cyc_pat.segment.map(dict(enumerate(SEGMENTS)))
cyc_pat["length_class"] = pd.cut(cyc_pat.length, [0, 2, 6, 99], labels=["2", "3-6", "7+"])
ceiling = cyc_pat.groupby("segment_name").agg(
    attempts=("length", "size"), len_3_6=("length", lambda s: s.between(3, 6).sum()),
    strict=("time_respecting_strict", "sum"), all_ach=("all_ach", "sum"),
    rule_ceiling=("length", lambda s: (s.between(3, 6) & cyc_pat.loc[s.index, "time_respecting_strict"]
                                        & cyc_pat.loc[s.index, "all_ach"] & cyc_pat.loc[s.index, "single_segment"]).sum()))
ceiling.to_csv(OUT / "ps2_ceiling.csv")
display(ceiling)
display(cyc_pat.groupby("length_class", observed=False).size().rename("CYCLE attempts by length").to_frame().T)

dev_dur = cyc_pat[(cyc_pat.segment == 0) & cyc_pat.time_respecting_strict.astype(bool)].duration_h.astype(float)
if len(dev_dur) >= 5:
    DELTA_H = float(math.ceil(np.quantile(dev_dur, CFG["delta_quantile"])))
    delta_note = f"p{int(CFG['delta_quantile']*100)} of {len(dev_dur)} time-respecting CYCLE attempts closing in dev"
elif cyc_pat.time_respecting_strict.astype(bool).any():
    DELTA_H = float(math.ceil(cyc_pat[cyc_pat.time_respecting_strict.astype(bool)].duration_h.max()))
    delta_note = "fewer than 5 dev attempts: max duration over all time-respecting attempts (flagged)"
else:
    DELTA_H = 72.0
    delta_note = "no time-respecting CYCLE attempt found: fallback 72 h (flagged)"
DELTA_S = int(DELTA_H * 3600)
print(f"δ = {DELTA_H:.0f} h ({delta_note})")

Pattern rows: 3,209; matched to graph transactions: 3,198 (the rest are self-loops/duplicates outside the graph)


,attempts,len_3_6,strict,all_ach,rule_ceiling
segment_name,,,,,
dev,18,6,16,18,5
tail,11,2,9,11,0
test,25,12,24,25,5


length_class,2,3-6,7+
CYCLE attempts by length,14,20,20


δ = 89 h (p95 of 16 time-respecting CYCLE attempts closing in dev)


## 2–4. SCC, liệt kê chu trình và lọc theo thời gian
Thuật toán (đã kiểm chứng với vét cạn mọi tổ hợp cạnh trên đồ thị nhỏ, ô kế tiếp):
- **Liệt kê**: mỗi chu trình đơn được tìm đúng một lần, gốc tại tài khoản có chỉ số nhỏ nhất; chỉ đi qua đỉnh lớn hơn gốc, trong cùng SCC, và chỉ khi còn quay về gốc kịp trong số bước còn lại (BFS ngược từ gốc).
- **Thứ tự thời gian**: với mỗi cách xoay vòng và mỗi thời điểm bắt đầu, đi tham lam theo cạnh sớm nhất hợp lệ để ra thời điểm đóng vòng sớm nhất; nếu ≤ δ thì đi lùi từ hạn chót để lấy thời điểm muộn nhất ở mỗi bước. Mọi cạnh song song nằm giữa hai mốc đó đều thuộc một dãy hợp lệ.

In [3]:
import time
from bisect import bisect_left, bisect_right

import numpy as np


def enumerate_cycles(adj_out, adj_in, comp, l_min, l_max, max_cycles, deadline, per_root_cap=10**9):
    """Simple directed cycles with l_min..l_max edges, each reported once (rooted at its smallest node).

    adj_out/adj_in: lists of sorted neighbour lists over local nodes 0..n-1.
    comp: SCC id per node; a cycle never leaves its SCC.
    per_root_cap bounds the cycles reported from one root, so a hub cannot exhaust the budget.
    Returns (cycles, roots_done, truncated_reason, roots_capped).
    """
    n = len(adj_out)
    cycles = []
    capped = 0
    for s in range(n):
        if time.monotonic() > deadline:
            return cycles, s, "time", capped
        cs = comp[s]
        # reverse BFS: distance (in edges) from y back to s, over nodes > s in the same SCC
        dist = {s: 0}
        frontier = [s]
        for d in range(1, l_max):
            nxt = []
            for x in frontier:
                for y in adj_in[x]:
                    if y > s and y not in dist and comp[y] == cs:
                        dist[y] = d
                        nxt.append(y)
            if not nxt:
                break
            frontier = nxt
        if len(dist) == 1 and s not in adj_out[s]:
            continue
        path = [s]
        onpath = {s}
        stack = [iter(adj_out[s])]
        found = 0
        while stack:
            it = stack[-1]
            k = len(path)
            moved = False
            for y in it:
                if y == s:
                    if k >= l_min:
                        cycles.append(tuple(path))
                        found += 1
                        if len(cycles) >= max_cycles:
                            return cycles, s, "max_cycles", capped
                    continue
                dy = dist.get(y)
                if dy is not None and y not in onpath and k + dy <= l_max:
                    path.append(y)
                    onpath.add(y)
                    stack.append(iter(adj_out[y]))
                    moved = True
                    break
            if not moved or found >= per_root_cap:
                if found >= per_root_cap:
                    capped += 1
                    break
                stack.pop()
                onpath.discard(path.pop())
    return cycles, n, "", capped


def temporal_check(steps, strict, delta):
    """steps: list of (times_list, ids_list) for the L pairs of a cycle in cycle order, times sorted.

    Tries every rotation and every start time. A chain is valid if times increase along the cycle
    (strictly if `strict`) and last - first <= delta.
    Returns (valid, flagged_ids:set, witness) where witness = (start_t, end_t, [ids]) of the
    earliest-closing valid chain.
    """
    L = len(steps)
    nxt = bisect_right if strict else bisect_left       # first index with time > / >= cur
    prv = bisect_left if strict else bisect_right       # count of times < / <= bound
    flagged = set()
    witness = None
    for r in range(L):
        st = steps[r:] + steps[:r]
        t0, id0 = st[0]
        last_start = None
        for a_idx, a in enumerate(t0):
            if a == last_start:
                continue
            last_start = a
            cur = a
            lo_idx = [a_idx]
            ok = True
            for i in range(1, L):
                ti = st[i][0]
                j = nxt(ti, cur)
                if j == len(ti):
                    ok = False
                    break
                cur = ti[j]
                lo_idx.append(j)
            if not ok:
                break            # later starts cannot complete either
            if cur - a > delta:
                continue
            D = a + delta
            hi_idx = [0] * L
            bound = D
            feasible = True
            for i in range(L - 1, 0, -1):
                ti = st[i][0]
                # last index with time <= bound (for the closing step) or < next step time
                if i == L - 1:
                    h = bisect_right(ti, bound) - 1
                else:
                    h = prv(ti, bound) - 1
                if h < lo_idx[i]:
                    feasible = False
                    break
                hi_idx[i] = h
                bound = ti[h]
            if not feasible:
                continue
            # step 0: every edge at the start time
            j0 = bisect_left(t0, a)
            j1 = bisect_right(t0, a)
            flagged.update(id0[j0:j1])
            for i in range(1, L):
                flagged.update(st[i][1][lo_idx[i]:hi_idx[i] + 1])
            if witness is None or cur < witness[1]:
                witness = (a, cur, [id0[a_idx]] + [st[i][1][lo_idx[i]] for i in range(1, L)])
    return witness is not None, flagged, witness


# Self-test against brute force on random small multigraphs (parallel edges, ties, all rotations).
import itertools, random
def _brute(edges, l_max, strict, delta):
    by = {}
    for u, v, t, i in edges:
        by.setdefault((u, v), []).append((t, i))
    nodes = sorted({x for e in edges for x in e[:2]})
    flagged = set()
    for L in range(2, l_max + 1):
        for perm in itertools.permutations(nodes, L):
            if perm[0] != min(perm):
                continue
            pairs = [(perm[k], perm[(k + 1) % L]) for k in range(L)]
            if not all(p in by for p in pairs):
                continue
            for r in range(L):
                pr = pairs[r:] + pairs[:r]
                for combo in itertools.product(*[by[p] for p in pr]):
                    ts_ = [x[0] for x in combo]
                    if all((ts_[k] < ts_[k + 1]) if strict else (ts_[k] <= ts_[k + 1]) for k in range(L - 1)) \
                            and ts_[-1] - ts_[0] <= delta:
                        flagged.update(x[1] for x in combo)
    return flagged
def _ours(edges, l_max, strict, delta):
    nodes = sorted({x for e in edges for x in e[:2]}); ix = {v: k for k, v in enumerate(nodes)}; n = len(nodes)
    pairs = sorted({(ix[u], ix[v]) for u, v, _, _ in edges}); a = np.array(pairs)
    _, comp = connected_components(coo_matrix((np.ones(len(a)), (a[:, 0], a[:, 1])), shape=(n, n)),
                                   directed=True, connection="strong")
    out = [[] for _ in range(n)]; inn = [[] for _ in range(n)]
    for x, y in pairs:
        if comp[x] == comp[y]:
            out[x].append(y); inn[y].append(x)
    cycles, _, _, _ = enumerate_cycles(out, inn, comp, 2, l_max, 10**9, time.monotonic() + 60)
    by = {}
    for u, v, t, i in sorted(edges, key=lambda e: (e[2], e[3])):
        by.setdefault((ix[u], ix[v]), ([], [])); by[(ix[u], ix[v])][0].append(t); by[(ix[u], ix[v])][1].append(i)
    flagged = set()
    for c in cycles:
        ok, f, _ = temporal_check([by[(c[k], c[(k + 1) % len(c)])] for k in range(len(c))], strict, delta)
        flagged |= f
    return flagged
_rng = random.Random(1)
for _ in range(150):
    n = _rng.randint(3, 6)
    E = [(_rng.randrange(n), _rng.randrange(n), _rng.randint(0, 6), k) for k in range(_rng.randint(3, 12))]
    E = [e for e in E if e[0] != e[1]]
    if E:
        for strict in (True, False):
            for delta in (2, 4, 10**9):
                assert _brute(E, 4, strict, delta) == _ours(E, 4, strict, delta), E
print("Brute-force checks passed (parallel edges, same-minute ties, rotations, δ).")

Brute-force checks passed (parallel edges, same-minute ties, rotations, δ).


In [4]:
edges = db.execute("""
    SELECT transaction_id, source_id, target_id, timestamp_s, payment_format = 'ACH' AS is_ach, usd_paid
    FROM read_parquet(?) WHERE timestamp_s >= ? AND timestamp_s < ?
""", [str(EDGES_FILE), int(seg_bounds[0]), int(seg_bounds[-1])]).df()
E_src = edges.source_id.to_numpy(np.int64); E_dst = edges.target_id.to_numpy(np.int64)
E_t = edges.timestamp_s.to_numpy(np.int64); E_id = edges.transaction_id.to_numpy(np.int64)
E_ach = edges.is_ach.to_numpy(bool); E_usd = edges.usd_paid.to_numpy(float)
E_seg = seg_of(E_t)
usd_of = dict(zip(E_id.tolist(), E_usd.tolist()))
print(f"Edges in main period + tail: {len(edges):,}")

VARIANTS = {"strict_delta": (True, DELTA_S), "nonstrict_delta": (False, DELTA_S), "strict_nodelta": (True, 10**12)}

def run_view(view, seg, l_max):
    t_start = time.monotonic()
    m = (E_seg == seg) & (E_ach if view == "ACH" else True)
    u, v, t, ids = E_src[m], E_dst[m], E_t[m], E_id[m]
    if len(u) == 0:
        return None
    nodes, inv = np.unique(np.r_[u, v], return_inverse=True)
    lu, lv = inv[:len(u)], inv[len(u):]
    n = len(nodes)
    pk = lu * n + lv
    order = np.lexsort((ids, t, pk))
    pk_s, t_s, id_s = pk[order], t[order], ids[order]
    upk, first = np.unique(pk_s, return_index=True)
    last = np.r_[first[1:], len(pk_s)]
    pu, pv = upk // n, upk % n
    _, comp = connected_components(coo_matrix((np.ones(len(pu)), (pu, pv)), shape=(n, n)), directed=True, connection="strong")
    sizes = np.bincount(comp)
    keep = (comp[pu] == comp[pv]) & (sizes[comp[pu]] >= 2)
    # SCC baseline: every edge whose two ends lie in the same non-trivial SCC
    scc_edge_ids = set()
    for a_, b_ in zip(first[keep].tolist(), last[keep].tolist()):
        scc_edge_ids.update(id_s[a_:b_].tolist())
    # Root order: accounts with fewer counterparties first, so each cycle is found from its quietest member
    # and a hub cannot use up the budget before the other accounts are visited.
    deg = np.bincount(pu[keep], minlength=n) + np.bincount(pv[keep], minlength=n)
    rank = np.empty(n, np.int64); rank[np.lexsort((np.arange(n), deg))] = np.arange(n)
    back = np.argsort(rank)                          # rank -> local node
    comp_r = comp[back]
    out = [[] for _ in range(n)]; inn = [[] for _ in range(n)]
    for x, y in zip(rank[pu[keep]].tolist(), rank[pv[keep]].tolist()):
        out[x].append(y); inn[y].append(x)
    for lst in out: lst.sort()
    for lst in inn: lst.sort()
    limit = CFG["views"][view]["time_limit_s"]
    deadline = t_start + limit                      # whole run; enumeration may use at most half of it
    cycles_r, roots_done, trunc, roots_capped = enumerate_cycles(out, inn, comp_r, 2, l_max, CFG["max_cycles"],
                                                                 t_start + limit / 2, CFG["per_root_cap"])
    cycles = [tuple(int(back[x]) for x in c) for c in cycles_r]
    pair_rng = {int(k): (int(a_), int(b_)) for k, a_, b_ in zip(upk[keep], first[keep], last[keep])}
    cache = {}
    def step(x, y):
        k = x * n + y
        if k not in cache:
            a_, b_ = pair_rng[k]
            cache[k] = (t_s[a_:b_].tolist(), id_s[a_:b_].tolist())
        return cache[k]
    recs = []
    flagged = {name: {} for name in VARIANTS}       # variant -> length -> set(ids)
    static = {}                                     # length -> set(ids)
    n_static_by_len = {}
    checked = 0
    for c in cycles:
        if time.monotonic() > deadline:
            trunc = trunc or "time"
            break
        L = len(c)
        steps = [step(c[k], c[(k + 1) % L]) for k in range(L)]
        static.setdefault(L, set()).update(i for s_ in steps for i in s_[1])
        rec = {"length": L, "accounts": " > ".join(str(int(nodes[x])) for x in c)}
        for name, (strict, delta) in VARIANTS.items():
            ok, f, w = temporal_check(steps, strict, delta)
            rec[name] = ok
            if ok:
                flagged[name].setdefault(L, set()).update(f)
                if name == "strict_delta":
                    rec.update({"start_ts": w[0], "end_ts": w[1], "witness_ids": " ".join(map(str, w[2])),
                                "witness_usd": " ".join(f"{usd_of[i]:.0f}" for i in w[2])})
        if rec["strict_delta"] or rec["nonstrict_delta"] or rec["strict_nodelta"]:
            recs.append(rec)                         # keep only time-respecting cycles on disk
        n_static_by_len[L] = n_static_by_len.get(L, 0) + 1
        checked += 1
    log = {"view": view, "segment": SEGMENTS[seg], "edges": int(m.sum()), "pairs": int(len(upk)),
           "scc_nodes": int((sizes[comp] >= 2).sum()), "largest_scc": int(sizes.max()),
           "static_cycles": len(cycles), "cycles_checked": checked, "temporal_cycles_saved": len(recs),
           "roots_done": int(roots_done), "roots_total": n, "roots_hitting_cap": int(roots_capped),
           "static_by_length": json.dumps(dict(sorted(n_static_by_len.items()))), "truncated": trunc, "seconds": round(time.monotonic() - t_start, 1)}
    return {"cycles": pd.DataFrame(recs), "flagged": flagged, "static": static, "scc": scc_edge_ids, "log": log}

results, logs = {}, []
for view in ("ACH", "ALL"):
    for seg in range(len(SEGMENTS)):
        r = run_view(view, seg, CFG["views"][view]["l_max"])
        if r is None:
            continue
        results[(view, seg)] = r
        logs.append(r["log"])
        print(r["log"], flush=True)
        cyc = r["cycles"]
        if len(cyc):
            cyc.insert(0, "cycle_id", np.arange(len(cyc)))
            cyc.to_csv(OUT / f"ps2_cycles_{view}_{SEGMENTS[seg]}.csv", index=False)
run_log = pd.DataFrame(logs)
run_log.to_csv(OUT / "ps2_run_log.csv", index=False)
display(run_log)
if run_log.truncated.astype(bool).any() or (run_log.roots_hitting_cap > 0).any():
    print("⚠ Một số lượt chạm giới hạn (số chu trình, thời gian, hoặc số chu trình mỗi gốc): số chu trình và "
          "số giao dịch bị gắn cờ của các lượt đó là cận dưới. Xem cột truncated / roots_hitting_cap.")

Edges in main period + tail: 4,487,124
{'view': 'ACH', 'segment': 'dev', 'edges': 298120, 'pairs': 163258, 'scc_nodes': 2227, 'largest_scc': 10, 'static_cycles': 1249, 'cycles_checked': 1249, 'temporal_cycles_saved': 1206, 'roots_done': 209931, 'roots_total': 209931, 'roots_hitting_cap': 0, 'static_by_length': '{"2": 1145, "3": 89, "4": 9, "5": 1, "6": 1, "7": 1, "10": 3}', 'truncated': '', 'seconds': 2.2}
{'view': 'ACH', 'segment': 'test', 'edges': 230611, 'pairs': 115185, 'scc_nodes': 1585, 'largest_scc': 12, 'static_cycles': 884, 'cycles_checked': 884, 'temporal_cycles_saved': 849, 'roots_done': 156284, 'roots_total': 156284, 'roots_hitting_cap': 0, 'static_by_length': '{"2": 800, "3": 53, "4": 18, "5": 6, "6": 4, "7": 2, "12": 1}', 'truncated': '', 'seconds': 3.2}
{'view': 'ACH', 'segment': 'tail', 'edges': 715, 'pairs': 645, 'scc_nodes': 11, 'largest_scc': 3, 'static_cycles': 9, 'cycles_checked': 9, 'temporal_cycles_saved': 7, 'roots_done': 716, 'roots_total': 716, 'roots_hitting_

,view,segment,edges,pairs,scc_nodes,largest_scc,static_cycles,cycles_checked,temporal_cycles_saved,roots_done,roots_total,roots_hitting_cap,static_by_length,truncated,seconds
0,ACH,dev,298120,163258,2227,10,1249,1249,1206,209931,209931,0,"{""2"": 1145, ""3"": 89, ""4"": 9, ""5"": 1, ""6"": 1, ""...",,2.2
1,ACH,test,230611,115185,1585,12,884,884,849,156284,156284,0,"{""2"": 800, ""3"": 53, ""4"": 18, ""5"": 6, ""6"": 4, ""...",,3.2
2,ACH,tail,715,645,11,3,9,9,7,716,716,0,"{""2"": 7, ""3"": 2}",,0.0
3,ALL,dev,2487578,569020,7676,4578,1367,1367,1316,406927,406927,0,"{""2"": 1249, ""3"": 97, ""4"": 11, ""5"": 5, ""6"": 5}",,8.6
4,ALL,test,1998831,424106,1705,15,941,941,907,369458,369458,0,"{""2"": 859, ""3"": 53, ""4"": 18, ""5"": 6, ""6"": 5}",,6.0
5,ALL,tail,715,645,11,3,9,9,7,716,716,0,"{""2"": 7, ""3"": 2}",,0.1


## 5. Đánh giá cấp giao dịch
Mẫu số = **mọi giao dịch của đoạn trong file gốc** (kể cả self-loop và dòng trùng). Một giao dịch thuộc nhiều vòng chỉ tính một lần. Nhãn chỉ được đọc từ đây.

In [5]:
labels = db.execute("""
    SELECT transaction_id, CAST(epoch(timestamp) AS BIGINT) AS ts, payment_format = 'ACH' AS is_ach, is_laundering
    FROM read_parquet(?)
    UNION ALL
    SELECT transaction_id, CAST(epoch(timestamp) AS BIGINT), payment_format = 'ACH', is_laundering FROM read_parquet(?)
""", [str(TXN_FILE), str(REMOVED_FILE)]).df()
labels["seg"] = seg_of(labels.ts.to_numpy())
lab = pd.Series(labels.is_laundering.to_numpy(), index=labels.transaction_id.to_numpy())

def wilson(tp, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = tp / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def union_len(d, lo, hi):
    s = set()
    for L, ids in d.items():
        if lo <= L <= hi:
            s |= ids
    return s

rng = np.random.default_rng(CFG["seed"])
rows = []
lo_main, hi_main = CFG["main_lengths"]
for seg in range(len(SEGMENTS)):
    ls = labels[labels.seg == seg]
    n_tx, n_ill = len(ls), int(ls.is_laundering.sum())
    ach = ls[ls.is_ach]; n_ach, tp_ach = len(ach), int(ach.is_laundering.sum())
    base = {"segment": SEGMENTS[seg], "n_transactions": n_tx, "n_laundering": n_ill}
    def add(method, view, ids):
        ids = np.fromiter(ids, np.int64) if not isinstance(ids, np.ndarray) else ids
        k = len(ids); tp = int(lab.reindex(ids).fillna(0).sum()) if k else 0
        lo, hi = wilson(tp, k)
        rows.append({**base, "view": view, "method": method, "flagged": k, "tp": tp,
                     "precision": tp / k if k else np.nan, "wilson_lo": lo, "wilson_hi": hi,
                     "recall": tp / n_ill if n_ill else np.nan,
                     "lift_vs_random": (tp / k) / (n_ill / n_tx) if k and n_ill else np.nan})
        return k
    rows.append({**base, "view": "-", "method": "Random (expected)", "flagged": np.nan, "tp": np.nan,
                 "precision": n_ill / n_tx if n_tx else np.nan, "recall": np.nan, "lift_vs_random": 1.0})
    lo_, hi_ = wilson(tp_ach, n_ach)
    rows.append({**base, "view": "ACH", "method": "ACH-only", "flagged": n_ach, "tp": tp_ach,
                 "precision": tp_ach / n_ach if n_ach else np.nan, "wilson_lo": lo_, "wilson_hi": hi_,
                 "recall": tp_ach / n_ill if n_ill else np.nan,
                 "lift_vs_random": (tp_ach / n_ach) / (n_ill / n_tx) if n_ach and n_ill else np.nan})
    for view in ("ACH", "ALL"):
        r = results.get((view, seg))
        if r is None:
            continue
        k_main = add("PS2 temporal cycles 3-6 (strict, ≤δ) [MAIN]" if view == "ACH" else "PS2 temporal cycles 3-6 (strict, ≤δ)",
                     view, union_len(r["flagged"]["strict_delta"], lo_main, hi_main))
        add("temporal 3-6, non-decreasing (≤), ≤δ", view, union_len(r["flagged"]["nonstrict_delta"], lo_main, hi_main))
        add("temporal 3-6, strict, no δ", view, union_len(r["flagged"]["strict_nodelta"], lo_main, hi_main))
        add("temporal length 2 (strict, ≤δ)", view, union_len(r["flagged"]["strict_delta"], 2, 2))
        if CFG["views"][view]["l_max"] > 6:
            add("temporal length 7-12 (strict, ≤δ)", view, union_len(r["flagged"]["strict_delta"], 7, 12))
        add("B: static cycles 3-6 (no time order)", view, union_len(r["static"], lo_main, hi_main))
        add("B: both ends in same SCC", view, r["scc"])
        if view == "ACH" and n_ach and k_main:
            draw = rng.choice(n_ach, min(k_main, n_ach), replace=False)
            add("B: random ACH sample (same size as MAIN)", view, ach.transaction_id.to_numpy()[draw])
tx_metrics = pd.DataFrame(rows)
tx_metrics.to_csv(OUT / "ps2_transaction_metrics.csv", index=False)
pd.set_option("display.max_colwidth", 60)
display(tx_metrics[tx_metrics.segment == "test"].drop(columns=["segment", "n_transactions", "n_laundering"]).round(5))

,view,method,flagged,tp,precision,recall,lift_vs_random,wilson_lo,wilson_hi
16,-,Random (expected),NaN,NaN,0.00110,NaN,1.00000,NaN,NaN
17,ACH,ACH-only,262486.0,1946.0,0.00741,0.86489,6.72850,0.00709,0.00775
18,ACH,"PS2 temporal cycles 3-6 (strict, ≤δ) [MAIN]",122.0,67.0,0.54918,0.02978,498.42093,0.46073,0.63463
19,ACH,"temporal 3-6, non-decreasing (≤), ≤δ",122.0,67.0,0.54918,0.02978,498.42093,0.46073,0.63463
20,ACH,"temporal 3-6, strict, no δ",143.0,79.0,0.55245,0.03511,501.38618,0.47064,0.63151
21,ACH,"temporal length 2 (strict, ≤δ)",1694.0,123.0,0.07261,0.05467,65.89812,0.06120,0.08596
22,ACH,"temporal length 7-12 (strict, ≤δ)",7.0,7.0,1.00000,0.00311,907.57244,0.64566,1.00000
23,ACH,B: static cycles 3-6 (no time order),210.0,92.0,0.43810,0.04089,397.60317,0.37270,0.50572
24,ACH,B: both ends in same SCC,1826.0,189.0,0.10350,0.08400,93.93822,0.09036,0.11832
25,ACH,B: random ACH sample (same size as MAIN),122.0,1.0,0.00820,0.00044,7.43912,0.00145,0.04497


## 6. Cấp pattern: tìm lại trọn attempt CYCLE
Một attempt CYCLE được **tìm lại trọn** nếu mọi giao dịch của nó (trong graph) đều bị rule chính gắn cờ. Bảng cũng cho biết các giao dịch rửa tiền bị gắn cờ thuộc typology nào: rule có thật sự bắt CYCLE hay bắt nhầm các typology khác có vòng.

In [6]:
prow = []
for seg in range(len(SEGMENTS)):
    r = results.get(("ACH", seg))
    flagged_main = union_len(r["flagged"]["strict_delta"], lo_main, hi_main) if r else set()
    flagged_any = union_len(r["flagged"]["strict_delta"], 2, 12) if r else set()
    for aid, g in pmap[pmap.typology == "CYCLE"].groupby("attempt_id"):
        if cyc_pat.loc[aid, "segment"] != seg:
            continue
        ids = set(g.transaction_id.tolist())
        prow.append({"segment": SEGMENTS[seg], "attempt_id": aid, "length": int(cyc_pat.loc[aid, "length"]),
                     "all_ach": cyc_pat.loc[aid, "all_ach"], "strict_in_patterns": cyc_pat.loc[aid, "time_respecting_strict"],
                     "recovered_main_3_6": ids <= flagged_main, "recovered_any_length": ids <= flagged_any,
                     "share_flagged_main": len(ids & flagged_main) / len(ids)})
pattern_rec = pd.DataFrame(prow)
pattern_rec.to_csv(OUT / "ps2_pattern_recovery.csv", index=False)
if len(pattern_rec):
    summary = pattern_rec.groupby("segment").agg(
        cycle_attempts=("attempt_id", "size"),
        recovered_main=("recovered_main_3_6", "sum"),
        recovered_any_length=("recovered_any_length", "sum"),
        partially=("share_flagged_main", lambda s: int(((s > 0) & (s < 1)).sum())))
    display(summary)

typ_rows = []
for seg in range(len(SEGMENTS)):
    r = results.get(("ACH", seg))
    if not r:
        continue
    ids = np.fromiter(union_len(r["flagged"]["strict_delta"], lo_main, hi_main), np.int64)
    pos = ids[lab.reindex(ids).fillna(0).to_numpy() == 1]
    t = pmap[pmap.transaction_id.isin(pos)].typology.value_counts()
    typ_rows.append({"segment": SEGMENTS[seg], "flagged_laundering": len(pos),
                     "not_in_any_listed_attempt": int(len(pos) - pmap.transaction_id.isin(pos).sum()), **t.to_dict()})
display(pd.DataFrame(typ_rows).fillna(0))

,cycle_attempts,recovered_main,recovered_any_length,partially
segment,,,,
dev,18,6,15,0
tail,11,0,0,0
test,25,3,7,2


,segment,flagged_laundering,not_in_any_listed_attempt,CYCLE,GATHER-SCATTER,RANDOM,FAN-IN,FAN-OUT,STACK,SCATTER-GATHER,BIPARTITE
0,dev,68,0,25.0,12.0,9.0,7.0,6.0,5.0,4.0,0.0
1,test,67,2,18.0,17.0,6.0,5.0,6.0,5.0,5.0,3.0
2,tail,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7. Kết luận tự động

In [7]:
def pick(seg, method_prefix, view="ACH"):
    d = tx_metrics[(tx_metrics.segment == seg) & (tx_metrics.view == view) & tx_metrics.method.str.startswith(method_prefix)]
    return d.iloc[0] if len(d) else None

def wins(seg):
    main = pick(seg, "PS2 temporal cycles 3-6 (strict, ≤δ) [MAIN]")
    ach, stat, scc = pick(seg, "ACH-only"), pick(seg, "B: static"), pick(seg, "B: both ends")
    if main is None or not main.flagged:
        return False, False, main
    beat_ach = main.wilson_lo > ach.precision
    beat_struct = main.precision > stat.precision and main.precision > scc.precision
    return bool(beat_ach), bool(beat_struct), main

t_ach, t_struct, m_test = wins("test")
d_ach, d_struct, m_dev = wins("dev")
tp_test = 0 if m_test is None or pd.isna(m_test.tp) else int(m_test.tp)
if tp_test < CFG["min_positives"]:
    verdict, reason = "Dừng", f"chỉ {tp_test} giao dịch rửa tiền bị gắn cờ trên test (< {CFG['min_positives']})"
elif t_ach and t_struct and d_ach and d_struct:
    verdict, reason = "Đào sâu", "thắng ACH-only, chu trình tĩnh và SCC trên test, và cũng thắng trên dev"
elif t_ach:
    verdict, reason = "Điều chỉnh", ("thắng ACH-only nhưng không thắng đối chứng cấu trúc" if not t_struct
                                     else "thắng trên test nhưng không ổn định trên dev")
else:
    verdict, reason = "Dừng", "không thắng ACH-only (cận dưới Wilson ≤ precision ACH-only)"

def fmt(row):
    return "—" if row is None or pd.isna(row.precision) else f"{row.precision:.2%} (recall {row.recall:.2%}, {int(row.flagged):,} cảnh báo)"
ceil_test = ceiling.loc["test"] if "test" in ceiling.index else None
decision = {
    "verdict": verdict, "reason": reason, "delta_hours": DELTA_H, "delta_rule": delta_note,
    "test_main": fmt(m_test), "test_ach_only": fmt(pick("test", "ACH-only")),
    "test_static_cycles": fmt(pick("test", "B: static")), "test_same_scc": fmt(pick("test", "B: both ends")),
    "dev_main": fmt(m_dev),
    "problem_statement_early_signal": "22% precision, ~5% recall (ACH temporal cycles)",
    "test_cycle_attempts": int(ceil_test.attempts) if ceil_test is not None else 0,
    "test_rule_ceiling_attempts": int(ceil_test.rule_ceiling) if ceil_test is not None else 0,
    "truncated_runs": run_log[run_log.truncated.astype(bool) | (run_log.roots_hitting_cap > 0)][
        ["view", "segment", "truncated", "roots_done", "roots_total", "roots_hitting_cap"]].to_dict("records"),
}
(OUT / "ps2_decision.json").write_text(json.dumps(decision, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"KẾT LUẬN PS2: {verdict} — {reason}\n")
for k in ("delta_hours", "test_main", "test_ach_only", "test_static_cycles", "test_same_scc", "dev_main",
          "problem_statement_early_signal", "test_cycle_attempts", "test_rule_ceiling_attempts"):
    print(f"• {k}: {decision[k]}")
print("\nFiles in", OUT)
for f in sorted(OUT.iterdir()):
    print("  ", f.name)

KẾT LUẬN PS2: Đào sâu — thắng ACH-only, chu trình tĩnh và SCC trên test, và cũng thắng trên dev

• delta_hours: 89.0
• test_main: 54.92% (recall 2.98%, 122 cảnh báo)
• test_ach_only: 0.74% (recall 86.49%, 262,486 cảnh báo)
• test_static_cycles: 43.81% (recall 4.09%, 210 cảnh báo)
• test_same_scc: 10.35% (recall 8.40%, 1,826 cảnh báo)
• dev_main: 43.31% (recall 2.99%, 157 cảnh báo)
• problem_statement_early_signal: 22% precision, ~5% recall (ACH temporal cycles)
• test_cycle_attempts: 25
• test_rule_ceiling_attempts: 5

Files in D:\KLTN\IBM_HI_Small_EDA\reports\ps2
   ps2_ceiling.csv
   ps2_cycles_ACH_dev.csv
   ps2_cycles_ACH_tail.csv
   ps2_cycles_ACH_test.csv
   ps2_cycles_ALL_dev.csv
   ps2_cycles_ALL_tail.csv
   ps2_cycles_ALL_test.csv
   ps2_decision.json
   ps2_pattern_recovery.csv
   ps2_pattern_transaction_map.csv
   ps2_run_log.csv
   ps2_transaction_metrics.csv
